# Invoice & Receipt Extraction — Colab Runner

Run the **OCR × LLM evaluation** and the **Streamlit web demo** on Google Colab (CPU or GPU).

This notebook keeps **no logic inline** — it loads the `invoice_extractor` package from the
repository and calls into it, so the Colab run uses the exact same code as local.

**Steps:** 1️⃣ Configure → 2️⃣ Clone + install → 3️⃣ Detect GPU → 4️⃣ API key → 5️⃣ Dataset → 6️⃣ Run eval → 7️⃣ (optional) Web demo.

> **GPU runtime:** `Runtime → Change runtime type → Hardware accelerator → GPU`. CPU works too (just slower).

## 1️⃣ Configuration
Set your repository URL (and branch). Everything else is auto-detected.

In [ ]:
# @title Configuration { display-mode: "form" }
REPO_URL = "https://github.com/TTNamUS/invoices-and-receipts_ocr.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
PROJECT_DIR = "ocr"  # @param {type:"string"}

# Evaluation defaults (override below or at run time)
EVAL_SPLIT = "test"   # @param ["test", "valid", "train"]
EVAL_LIMIT = 5        # @param {type:"integer"}  # small = quick smoke test; 0/None = full split
EVAL_TYPE = "all"     # @param ["all", "invoice", "receipt"]
OCR_ENGINE = "paddleocr"  # @param ["paddleocr", "tesseract"]

print(f"Repo:   {REPO_URL} (branch={BRANCH})")
print(f"Eval:   split={EVAL_SPLIT} limit={EVAL_LIMIT} type={EVAL_TYPE} engine={OCR_ENGINE}")

## 2️⃣ Clone the repository & install
Clones the repo, installs the package editable (`pip install -e .`), plus the matching
**paddlepaddle** build (GPU or CPU).

In [ ]:
import os, subprocess, sys

# Detect GPU first so we install the right paddlepaddle wheel.
HAS_GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print("GPU detected:" , HAS_GPU)

# Clone (skip if already present)
if not os.path.isdir(PROJECT_DIR):
    !git clone --branch $BRANCH $REPO_URL $PROJECT_DIR
%cd $PROJECT_DIR

# paddlepaddle: GPU build needs CUDA-matched wheel; CPU build otherwise.
if HAS_GPU:
    # Colab GPUs typically run CUDA 12.x. Adjust the index URL if your runtime differs.
    !pip install -q paddlepaddle-gpu
else:
    !pip install -q paddlepaddle

# Install the project (pulls litellm, paddleocr, datasets, streamlit, …)
!pip install -q -e .
print("\n✅ Install complete.")

## 3️⃣ Configure device (CPU / GPU)
Sets `PADDLE_USE_GPU` from the detected hardware. The PaddleOCR engine reads this and falls
back to CPU automatically if the GPU build isn't usable.

In [ ]:
import os
os.environ["PADDLE_USE_GPU"] = "true" if HAS_GPU else "false"
os.environ["OCR_ENGINE"] = OCR_ENGINE
print("PADDLE_USE_GPU =", os.environ["PADDLE_USE_GPU"])

# Quick sanity: does paddle see the GPU?
try:
    import paddle
    print("paddle.is_compiled_with_cuda():", paddle.is_compiled_with_cuda())
except Exception as e:
    print("paddle not importable yet:", e)

## 4️⃣ API key & provider
Enter the key for your chosen provider. It's stored only in this runtime's environment.

In [ ]:
import os
from getpass import getpass

PROVIDER = "openai"  # @param ["openai", "gemini"]
os.environ["LLM_PROVIDER"] = PROVIDER

if PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")
    os.environ.setdefault("OPENAI_MODEL", "gpt-5.4-mini")
else:
    os.environ["GOOGLE_API_KEY"] = getpass("GOOGLE_API_KEY: ")
    os.environ.setdefault("GEMINI_MODEL", "gemini/gemini-3.5-flash")

print("Provider set:", PROVIDER)

## 5️⃣ Dataset
The evaluator loads a HuggingFace `load_from_disk` dataset at `settings.dataset_path`
(default `dataset/invoices-and-receipts_ocr_v1/`). On a fresh Colab we download it from the Hub
and save it to that path.

If you already have the dataset (e.g. via Google Drive), set `DATASET_PATH` to skip the download.

In [ ]:
import os
from pathlib import Path

DATASET_PATH = "dataset/invoices-and-receipts_ocr_v1"  # @param {type:"string"}
os.environ["DATASET_PATH"] = DATASET_PATH

if not Path(DATASET_PATH).exists():
    print("Downloading dataset from HuggingFace Hub … (a few minutes)")
    from datasets import load_dataset
    ds = load_dataset("mychen76/invoices-and-receipts_ocr_v1")
    Path(DATASET_PATH).parent.mkdir(parents=True, exist_ok=True)
    ds.save_to_disk(DATASET_PATH)
    print("Saved to", DATASET_PATH)
else:
    print("Dataset already present at", DATASET_PATH)

## 6️⃣ Run evaluation
Calls the same batch runner as the CLI. The report dict is returned in-process so you can
inspect it; it's also written to `output/evaluation_report.json`, and each document's LLM
extraction is saved to `output/extractions/<id>.json`.

In [ ]:
from invoice_extractor.config import settings
from invoice_extractor.evaluation.report import run_evaluation, report_to_dict

limit = EVAL_LIMIT if EVAL_LIMIT and EVAL_LIMIT > 0 else None
report = run_evaluation(
    split=EVAL_SPLIT,
    limit=limit,
    doc_type=EVAL_TYPE,
    engine=OCR_ENGINE,
    output_path=settings.eval_report_path,  # default: output/evaluation_report.json
)

In [ ]:
# Pretty-print the important-field metrics
import json
d = report_to_dict(report)
print(json.dumps({
    "total_files_evaluated": d["total_files_evaluated"],
    "total_invoices_evaluated": d["total_invoices_evaluated"],
    "invoice_metrics": d["invoice_metrics"],
    "receipt_metrics": d["receipt_metrics"],
}, indent=2))

### (Optional) Inspect a single image end-to-end
Use the shared `run_pipeline` orchestrator on one dataset sample.

In [ ]:
from datasets import load_from_disk
from invoice_extractor import run_pipeline
import json

ds = load_from_disk(DATASET_PATH)[EVAL_SPLIT]
sample = ds[0]
out = run_pipeline(sample["image"], engine=OCR_ENGINE, file_name=str(sample["id"]))

print("OCR avg confidence:", out["ocr"]["average_confidence"])
print("\n--- raw text (first 400 chars) ---\n", out["ocr"]["raw_text"][:400])
print("\n--- extraction ---\n", json.dumps(out["extraction"], indent=2, ensure_ascii=False)[:1200])

## 7️⃣ (Optional) Web demo via Streamlit
Runs the existing `web/app.py` and exposes a public URL through a tunnel.

1. Run the launcher cell below.
2. Click the printed `https://*.loca.lt` URL.
3. The tunnel password is the **public IP** printed by the cell (localtunnel asks for it once).

In [ ]:
# Tunnel password (your Colab public IP) — paste this when localtunnel prompts.
import urllib.request
print("Tunnel password (public IP):", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip())

In [ ]:
import os, subprocess, time

# Streamlit reads provider/keys/device from the env we set above.
# Launch the app in the background, then open a localtunnel on port 8501.
proc = subprocess.Popen(
    ["streamlit", "run", "web/app.py",
     "--server.port", "8501", "--server.headless", "true"],
    env={**os.environ},
    stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT,
)
time.sleep(6)
print("Streamlit starting (see streamlit.log). Opening tunnel …\n")
!npx --yes localtunnel --port 8501

---
**Troubleshooting**
- *Tunnel page asks for a password* → paste the public IP from the cell above.
- *GPU not used* → confirm `Runtime → GPU`, re-run cells 2–3; the engine logs `device=gpu`.
- *PaddleOCR GPU init fails* → it auto-falls back to CPU (you'll see a warning). The eval still runs.
- *Streamlit log* → `!cat streamlit.log` to debug startup errors.
- *Stop the demo* → interrupt the launcher cell, or run `proc.terminate()` in a new cell.